In [88]:
import pandas as pd
from prophet import Prophet
from DBD_1 import load_diesel
from DBD_1 import load_petrol
from DBD_1 import print_feature_dist
from DBD_1 import print_feature_vs_y_line_graphs
import numpy as np
from sklearn.metrics import root_mean_squared_error, r2_score, mean_absolute_error
from matplotlib import pyplot as plt


## Load Data

In [78]:
diesel_df = load_diesel()
diesel_df["Total_Production_Lag1"] = diesel_df["Total_Production"].shift(1)
petrol_df = load_petrol()
petrol_df["Total_Production_Lag1"] = petrol_df["Total_Production"].shift(1)
diesel_df.head()

/Users/hoyle/Documents/Uni/4th_Year/MOR441-AMX/fools-optimum/notebooks/DBD_1.py:51: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"])
/Users/hoyle/Documents/Uni/4th_Year/MOR441-AMX/fools-optimum/notebooks/DBD_1.py:51: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"])


,Date,Total_Fuel_Price,BFP,USDZAR_Mean,USDZAR_Std,USDZAR_LastWeekMean,Brent_MonthMean,Brent_MonthStd,Brent_LastWeekMean,Brent_LastWeekStd,...,GECON_Lag1,GECON_Lag2,GECON_Lag3,INDPRO_Lag1,INDPRO_Lag2,INDPRO_Lag3,INDPRO_Lag6,INDPRO_Lag12,Total_Production,Total_Production_Lag1
0,2011-01-01,789.451,477.03,6.905624,0.159159,7.088029,96.523500,1.432121,96.987143,0.932393,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,68537.7764,NaN
1,2011-02-01,821.451,509.03,7.140163,0.118266,7.115057,101.449655,4.976158,109.406667,4.277731,...,0.247128,NaN,NaN,93.4814,NaN,NaN,NaN,NaN,68700.0099,68537.7764
2,2011-03-01,884.451,572.03,6.967600,0.120730,6.867400,113.560000,3.168441,115.761429,0.556729,...,0.423471,0.247128,NaN,93.1102,93.4814,NaN,NaN,NaN,67169.5421,68700.0099
3,2011-04-01,953.851,616.03,6.805250,0.119489,6.745443,120.723704,4.066770,124.930000,1.233450,...,-0.558572,0.423471,0.247128,94.0788,93.1102,93.4814,NaN,NaN,67351.0479,67169.5421
4,2011-05-01,969.851,632.03,6.828940,0.132094,6.962600,116.875000,5.568645,114.035000,2.422369,...,-0.258231,-0.558572,0.423471,93.7749,94.0788,93.1102,NaN,NaN,67087.4645,67351.0479


## Feature Selection

In [ ]:

def evaluate_feature_set(df, feature_cols, target_col="Total_Fuel_Price",
                          n_windows=5, min_train_months=24):
    """
    Rolling walk-forward evaluation of a Prophet regressor set.
    Test window size is derived from TEST_RATIO applied to the usable data length;
    forecast target is shifted by FORECAST_HORIZON months (global).
    """
    prophet_df = df[["Date", target_col] + feature_cols].copy()
    prophet_df = prophet_df.rename(columns={"Date": "ds", target_col: "y"})
    prophet_df = prophet_df.sort_values("ds").reset_index(drop=True)

    # Target is shifted to prevent data leakage - IE model can't view data from after that month
    prophet_df["y"] = prophet_df["y"].shift(-FORECAST_HORIZON)
    prophet_df = prophet_df.dropna().reset_index(drop=True)

    n_total = len(prophet_df)
    if n_total < min_train_months + FORECAST_HORIZON:
        return np.nan

    # derive test window size from TEST_RATIO, applied to the full usable series
    window_size = max(FORECAST_HORIZON, int(round(n_total * TEST_RATIO / n_windows)))

    rmses = []
    for i in range(n_windows):
        end = n_total - i * window_size
        start = end - window_size
        if start < min_train_months:
            break

        train_fold = prophet_df.iloc[:start]
        test_fold = prophet_df.iloc[start:end]
        if len(test_fold) == 0:
            continue

        try:
            m = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
            for col in feature_cols:
                m.add_regressor(col)
            m.fit(train_fold)

            fc = m.predict(test_fold[["ds"] + feature_cols])
            rmse = root_mean_squared_error(test_fold["y"], fc["yhat"])
            rmses.append(rmse)
        except Exception:
            continue

    return np.mean(rmses) if rmses else np.nan


def select_prophet_features(df, candidate_features, target_col="Total_Fuel_Price",
                             n_windows=5, min_train_months=24, max_features=10):
    remaining = list(candidate_features)
    selected = []

    best_rmse = evaluate_feature_set(df, selected, target_col, n_windows, min_train_months)

    improved = True
    while improved and remaining and len(selected) < max_features:
        improved = False
        scores = {
            feat: evaluate_feature_set(df, selected + [feat], target_col, n_windows, min_train_months)
            for feat in remaining
        }

        best_feat = min(scores, key=lambda k: scores[k] if not np.isnan(scores[k]) else np.inf)
        best_candidate_rmse = scores[best_feat]

        if not np.isnan(best_candidate_rmse) and best_candidate_rmse < best_rmse:
            selected.append(best_feat)
            remaining.remove(best_feat)
            best_rmse = best_candidate_rmse
            improved = True

    return selected, best_rmse

In [101]:
# ---- Global configuration ----
TRAIN_RATIO = 0.8        # proportion of usable data used for training in each fold
TEST_RATIO = 0.2         # proportion used for testing (must sum to 1 with TRAIN_RATIO)
FORECAST_HORIZON = 4     # months ahead to forecast (e.g. 1 = one-month-ahead, 3 = three-months-ahead)

assert abs(TRAIN_RATIO + TEST_RATIO - 1.0) < 1e-9, "TRAIN_RATIO + TEST_RATIO must equal 1"

exclude_cols = ["Date", "Total_Production"]
candidate_features = [c for c in diesel_df.columns if c not in exclude_cols]

selected_diesel_features, final_rmse = select_prophet_features(diesel_df, candidate_features)

print(selected_diesel_features)
print(f"Final mean RMSE: {final_rmse:.4f}")

11:01:35 - cmdstanpy - INFO - Chain [1] start processing
11:01:35 - cmdstanpy - INFO - Chain [1] done processing
11:01:35 - cmdstanpy - INFO - Chain [1] start processing
11:01:35 - cmdstanpy - INFO - Chain [1] done processing
11:01:35 - cmdstanpy - INFO - Chain [1] start processing
11:01:35 - cmdstanpy - INFO - Chain [1] done processing
11:01:35 - cmdstanpy - INFO - Chain [1] start processing
11:01:35 - cmdstanpy - INFO - Chain [1] done processing
11:01:35 - cmdstanpy - INFO - Chain [1] start processing
11:01:35 - cmdstanpy - INFO - Chain [1] done processing
11:01:35 - cmdstanpy - INFO - Chain [1] start processing
11:01:35 - cmdstanpy - INFO - Chain [1] done processing
11:01:35 - cmdstanpy - INFO - Chain [1] start processing
11:01:35 - cmdstanpy - INFO - Chain [1] done processing
11:01:35 - cmdstanpy - INFO - Chain [1] start processing
11:01:35 - cmdstanpy - INFO - Chain [1] done processing
11:01:35 - cmdstanpy - INFO - Chain [1] start processing
11:01:35 - cmdstanpy - INFO - Chain [1]

['GSCPI_Lag12', 'BFP_Lag2', 'GPR_Lag6', 'USDZAR_Delta_Lag2']
Final mean RMSE: 254.6705


## Model evaluation

In [ ]:
def run_prophet_model(df, feature_cols, target_col="Total_Fuel_Price",
                       include_2026=True, plot=True):
    """
    Fits and evaluates a Prophet model using rolling-origin (walk-forward) validation.

    At each step, the model is trained on all data known up to that point, and
    predicts FORECAST_HORIZON months ahead. Once that actual outcome "arrives"
    (i.e. the loop advances past it), it's folded into the training set for the
    next refit — so the test set is walked forward one prediction at a time
    rather than forecast in one shot.

    Uses global TRAIN_RATIO / TEST_RATIO to define the initial train/test split
    point, and FORECAST_HORIZON for how many months ahead each prediction targets.

    Parameters
    ----------
    df           : master dataframe (e.g. master_diesel_df)
    feature_cols : list of selected regressor column names
    target_col   : target price column
    include_2026 : if False, drops all rows from 2026 onward before modelling
    plot         : if True, shows an actual vs predicted plot for the test set

    Returns
    -------
    dict with rmse, mae, r2, per-step predictions, and the final fitted model
    """
    feature_cols = [c for c in feature_cols if c not in ("Date", target_col)]

    data = df.copy()
    if not include_2026:
        data = data[data["Date"] < "2026-01-01"]

    prophet_df = data[["Date", target_col] + feature_cols].copy()
    prophet_df = prophet_df.rename(columns={"Date": "ds", target_col: "y"})
    prophet_df = prophet_df.sort_values("ds").reset_index(drop=True)

    # Only the target is shifted forward — features stay at row t
    prophet_df["y"] = prophet_df["y"].shift(-FORECAST_HORIZON)
    prophet_df = prophet_df.dropna().reset_index(drop=True)

    n_total = len(prophet_df)
    split_idx = int(round(n_total * TRAIN_RATIO))

    if split_idx >= n_total:
        raise ValueError("Test set is empty — check TRAIN_RATIO/TEST_RATIO and data length.")

    dates, y_true_list, y_pred_list, y_lower_list, y_upper_list = [], [], [], [], []

    # Walk forward one test point at a time, expanding the training window each step
    for i in range(split_idx, n_total):
        train_fold = prophet_df.iloc[:i]        # everything known up to (not including) row i
        test_row = prophet_df.iloc[[i]]          # single point being predicted this step

        model = Prophet(yearly_seasonality=True, weekly_seasonality=False, daily_seasonality=False)
        for col in feature_cols:
            model.add_regressor(col)
        model.fit(train_fold)

        fc = model.predict(test_row[["ds"] + feature_cols])

        dates.append(test_row["ds"].values[0])
        y_true_list.append(test_row["y"].values[0])
        y_pred_list.append(fc["yhat"].values[0])
        y_lower_list.append(fc["yhat_lower"].values[0])
        y_upper_list.append(fc["yhat_upper"].values[0])


    y_true = np.array(y_true_list)
    y_pred = np.array(y_pred_list)

    results_df = pd.DataFrame({
        "ds": dates,
        "y_true": y_true,
        "y_pred": y_pred,
        "yhat_lower": y_lower_list,
        "yhat_upper": y_upper_list,
    })

    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    print(f"include_2026={include_2026} | horizon={FORECAST_HORIZON}m | "
          f"walk-forward steps={len(results_df)}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAE:  {mae:.4f}")
    print(f"R2:   {r2:.4f}")

    if plot:
        fig, ax = plt.subplots(figsize=(11, 4))
        ax.plot(results_df["ds"], results_df["y_true"], label="Actual", marker="o")
        ax.plot(results_df["ds"], results_df["y_pred"], label="Predicted", marker="o")
        ax.fill_between(results_df["ds"], results_df["yhat_lower"], results_df["yhat_upper"], alpha=0.2)
        ax.legend()
        ax.set_title(f"Prophet — {target_col} rolling {FORECAST_HORIZON}m-ahead (include_2026={include_2026})")
        plt.tight_layout()
        plt.show()

    return {
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "model": model,          # final model, fit on all data up to the last test point
        "results_df": results_df,
        "train_df": train_fold
    }

In [121]:
for horizon, results in all_petrol_results.items():
    features = results["features"]
    print(f"Horizon: {horizon:<7} \n{features}")

Horizon: 1       
['BFP', 'BFP_Delta_Lag2', 'GECON', 'Brent_LastWeekMean', 'Brent_MonthMean_Lag1', 'Brent_LastWeekStd_Lag2', 'USDZAR_LastWeekMean', 'Brent_MonthMean', 'Brent_LastWeekMean_Lag2', 'GPR_Lag2']
Horizon: 2       
['BFP', 'BFP_Delta_Lag1', 'GSCPI_Lag12', 'GPR_Lag6', 'GECON', 'GECON_Lag3', 'GPR', 'BFP_Delta_Lag4']
Horizon: 3       
['BFP', 'GSCPI_Lag12', 'BFP_Delta_Lag1', 'GPR_Lag6', 'GECON', 'INDPRO', 'BFP_Lag2', 'Brent_LastWeekStd', 'BFP_Lag6', 'BFP_Delta_Lag3']
Horizon: 4       
['GSCPI_Lag12', 'BFP_Lag1', 'GPR_Lag6', 'BFP_Delta_Lag2', 'GECON', 'USDZAR_Std', 'Brent_LastWeekStd_Lag3', 'GECON_Lag1', 'Brent_MonthStd', 'USDZAR_Delta_Lag2']
Horizon: 5       
['GSCPI_Lag12', 'BFP', 'GPR_Lag3', 'BFP_Delta_Lag1', 'Brent_LastWeekStd_Lag3', 'Total_Production_Lag1']
Horizon: 6       
['GSCPI_Lag12', 'BFP', 'GPR_Lag3', 'BFP_Delta_Lag5', 'Brent_LastWeekStd_Lag2', 'GPR_Lag2', 'BFP_Delta_Lag4']


In [122]:
# Print all results at the end
print("\nSummary of Results:")
print("====================================================================")
print("| Horizon | 2026 included | RMSE     | MAE      | R^2      |")
print("====================================================================")
for horizon, results in all_petrol_results.items():
    for include_2026, result_key in zip([True, False], ["with_2026", "no_2026"]):
        result = results[result_key]
        print(f"| {horizon:<7} | {str(include_2026):<14} | {result['rmse']:<8.4f} | {result['mae']:<8.4f} | {result['r2']:<8.4f} |")
print("====================================================================")


Summary of Results:
| Horizon | 2026 included | RMSE     | MAE      | R^2      |
| 1       | True           | 51.3195  | 40.9421  | 0.9230   |
| 1       | False          | 53.0845  | 41.7182  | 0.8253   |
| 2       | True           | 137.9166 | 106.1398 | 0.4733   |
| 2       | False          | 115.4558 | 92.9729  | 0.2058   |
| 3       | True           | 166.4230 | 114.9279 | 0.2375   |
| 3       | False          | 124.1819 | 100.5920 | 0.0812   |
| 4       | True           | 179.5527 | 131.0247 | 0.1125   |
| 4       | False          | 121.6947 | 100.4320 | 0.1176   |
| 5       | True           | 173.5608 | 128.2364 | 0.1707   |
| 5       | False          | 116.6771 | 99.9001  | 0.1889   |
| 6       | True           | 165.4884 | 125.0380 | 0.2461   |
| 6       | False          | 104.4706 | 89.5593  | 0.3686   |


In [124]:
for horizon, results in all_diesel_results.items():
    features = results["features"]
    print(f"Horizon: {horizon:<7} \n{features}")
    

Horizon: 1       
['BFP', 'Brent_LastWeekMean', 'Brent_LastWeekMean_Lag2', 'Total_Production_Lag1', 'USDZAR_LastWeekMean', 'USDZAR_Mean_Lag1', 'BFP_Delta_Lag5', 'BFP_Delta_Lag1', 'BFP_Lag1', 'Brent_MonthMean_Lag1']
Horizon: 2       
['BFP', 'Brent_LastWeekMean', 'Brent_MonthMean_Lag1', 'BFP_Delta_Lag1', 'Brent_MonthStd', 'GSCPI_Lag12', 'USDZAR_LastWeekMean', 'GECON_Lag3', 'USDZAR_Mean_Lag1', 'Brent_MonthMean']
Horizon: 3       
['BFP', 'GSCPI_Lag12', 'GPR_Lag6', 'BFP_Delta_Lag1', 'Brent_LastWeekStd', 'GECON_Lag3', 'USDZAR_Delta_Lag1']
Horizon: 4       
['GSCPI_Lag12', 'BFP_Lag2', 'GPR_Lag6', 'USDZAR_Delta_Lag2']
Horizon: 5       
['GSCPI_Lag12']
Horizon: 6       
['GSCPI_Lag12', 'GECON', 'GPR_Lag6']


In [125]:
# Print all results at the end
print("\nSummary of Results:")
print("====================================================================")
print("| Horizon | 2026 included | RMSE     | MAE      | R^2      |")
print("====================================================================")
for horizon, results in all_diesel_results.items():
    for include_2026, result_key in zip([True, False], ["with_2026", "no_2026"]):
        result = results[result_key]
        print(f"| {horizon:<7} | {str(include_2026):<14} | {result['rmse']:<8.4f} | {result['mae']:<8.4f} | {result['r2']:<8.4f} |")
print("====================================================================")


Summary of Results:
| Horizon | 2026 included | RMSE     | MAE      | R^2      |
| 1       | True           | 116.8446 | 70.3609  | 0.8542   |
| 1       | False          | 62.2736  | 48.8523  | 0.8440   |
| 2       | True           | 193.3446 | 126.5257 | 0.6113   |
| 2       | False          | 119.7145 | 99.5984  | 0.4448   |
| 3       | True           | 317.5438 | 192.9060 | -0.0310  |
| 3       | False          | 185.9549 | 148.2243 | -0.3396  |
| 4       | True           | 319.5041 | 188.0659 | -0.0438  |
| 4       | False          | 208.4546 | 155.7920 | -0.6834  |
| 5       | True           | 295.0028 | 204.1233 | 0.1102   |
| 5       | False          | 195.5227 | 158.5327 | -0.4810  |
| 6       | True           | 302.2192 | 207.6060 | 0.0661   |
| 6       | False          | 196.0358 | 156.0651 | -0.4462  |


In [111]:
FORECAST_HORIZON = 1
select_diesel_features = ['BFP',
 'Brent_LastWeekMean',
 'Brent_LastWeekMean_Lag2',
 'Total_Production_Lag1',
 'USDZAR_LastWeekMean',
 'USDZAR_Mean_Lag1',
 'BFP_Delta_Lag5',
 'BFP_Delta_Lag1',
 'BFP_Lag1',
 'Brent_MonthMean_Lag1']
prophet_diesel_df = run_prophet_model(diesel_df, select_diesel_features, include_2026=True, plot=False)
print_feature_dist(prophet_diesel_df['train_df'][select_diesel_features])
print_feature_vs_y_line_graphs(prophet_diesel_df['train_df'])

11:16:36 - cmdstanpy - INFO - Chain [1] start processing
11:16:36 - cmdstanpy - INFO - Chain [1] done processing
11:16:36 - cmdstanpy - INFO - Chain [1] start processing
11:16:36 - cmdstanpy - INFO - Chain [1] done processing
11:16:36 - cmdstanpy - INFO - Chain [1] start processing
11:16:36 - cmdstanpy - INFO - Chain [1] done processing
11:16:36 - cmdstanpy - INFO - Chain [1] start processing
11:16:36 - cmdstanpy - INFO - Chain [1] done processing
11:16:36 - cmdstanpy - INFO - Chain [1] start processing
11:16:36 - cmdstanpy - INFO - Chain [1] done processing
11:16:36 - cmdstanpy - INFO - Chain [1] start processing
11:16:37 - cmdstanpy - INFO - Chain [1] done processing
11:16:37 - cmdstanpy - INFO - Chain [1] start processing
11:16:37 - cmdstanpy - INFO - Chain [1] done processing
11:16:37 - cmdstanpy - INFO - Chain [1] start processing
11:16:37 - cmdstanpy - INFO - Chain [1] done processing
11:16:37 - cmdstanpy - INFO - Chain [1] start processing
11:16:37 - cmdstanpy - INFO - Chain [1]

include_2026=True | horizon=1m | walk-forward steps=36
RMSE: 116.8446
MAE:  70.3609
R2:   0.8542


In [120]:
# Initialize a dictionary to store results for each horizon
all_petrol_results = {}

# Loop through forecast horizons from 1 to 6
for horizon in range(1, 7):
    print(f"Running Prophet model with FORECAST_HORIZON = {horizon}")
    FORECAST_HORIZON = horizon  # Update the global forecast horizon
    exclude_cols = ["Date", "Total_Production"]
    candidate_features = [c for c in petrol_df.columns if c not in exclude_cols]

    selected_petrol_features, final_rmse = select_prophet_features(petrol_df, candidate_features)

    # Run the model with and without 2026 data
    results_with_2026 = run_prophet_model(petrol_df, selected_petrol_features, include_2026=True, plot=False)
    results_no_2026 = run_prophet_model(petrol_df, selected_petrol_features, include_2026=False, plot=False)

    # Store the results in the dictionary
    all_petrol_results[horizon] = {
        "with_2026": results_with_2026,
        "no_2026": results_no_2026,
        "features": selected_petrol_features
    }

12:01:40 - cmdstanpy - INFO - Chain [1] start processing
12:01:40 - cmdstanpy - INFO - Chain [1] done processing


Running Prophet model with FORECAST_HORIZON = 1


12:01:40 - cmdstanpy - INFO - Chain [1] start processing
12:01:40 - cmdstanpy - INFO - Chain [1] done processing
12:01:40 - cmdstanpy - INFO - Chain [1] start processing
12:01:40 - cmdstanpy - INFO - Chain [1] done processing
12:01:41 - cmdstanpy - INFO - Chain [1] start processing
12:01:41 - cmdstanpy - INFO - Chain [1] done processing
12:01:41 - cmdstanpy - INFO - Chain [1] start processing
12:01:41 - cmdstanpy - INFO - Chain [1] done processing
12:01:41 - cmdstanpy - INFO - Chain [1] start processing
12:01:41 - cmdstanpy - INFO - Chain [1] done processing
12:01:41 - cmdstanpy - INFO - Chain [1] start processing
12:01:41 - cmdstanpy - INFO - Chain [1] done processing
12:01:41 - cmdstanpy - INFO - Chain [1] start processing
12:01:41 - cmdstanpy - INFO - Chain [1] done processing
12:01:41 - cmdstanpy - INFO - Chain [1] start processing
12:01:41 - cmdstanpy - INFO - Chain [1] done processing
12:01:41 - cmdstanpy - INFO - Chain [1] start processing
12:01:41 - cmdstanpy - INFO - Chain [1]

include_2026=True | horizon=1m | walk-forward steps=37
RMSE: 51.3195
MAE:  40.9421
R2:   0.9230


12:04:52 - cmdstanpy - INFO - Chain [1] start processing
12:04:52 - cmdstanpy - INFO - Chain [1] done processing
12:04:52 - cmdstanpy - INFO - Chain [1] start processing
12:04:52 - cmdstanpy - INFO - Chain [1] done processing
12:04:52 - cmdstanpy - INFO - Chain [1] start processing
12:04:52 - cmdstanpy - INFO - Chain [1] done processing
12:04:52 - cmdstanpy - INFO - Chain [1] start processing
12:04:52 - cmdstanpy - INFO - Chain [1] done processing
12:04:52 - cmdstanpy - INFO - Chain [1] start processing
12:04:52 - cmdstanpy - INFO - Chain [1] done processing
12:04:52 - cmdstanpy - INFO - Chain [1] start processing
12:04:52 - cmdstanpy - INFO - Chain [1] done processing
12:04:52 - cmdstanpy - INFO - Chain [1] start processing
12:04:52 - cmdstanpy - INFO - Chain [1] done processing
12:04:52 - cmdstanpy - INFO - Chain [1] start processing
12:04:52 - cmdstanpy - INFO - Chain [1] done processing
12:04:52 - cmdstanpy - INFO - Chain [1] start processing
12:04:52 - cmdstanpy - INFO - Chain [1]

...

include_2026=False | horizon=5m | walk-forward steps=33
RMSE: 116.6771
MAE:  99.9001
R2:   0.1889
Running Prophet model with FORECAST_HORIZON = 6


12:39:09 - cmdstanpy - INFO - Chain [1] start processing
12:39:09 - cmdstanpy - INFO - Chain [1] done processing
12:39:09 - cmdstanpy - INFO - Chain [1] start processing
12:39:09 - cmdstanpy - INFO - Chain [1] done processing
12:39:09 - cmdstanpy - INFO - Chain [1] start processing
12:39:09 - cmdstanpy - INFO - Chain [1] done processing
12:39:09 - cmdstanpy - INFO - Chain [1] start processing
12:39:09 - cmdstanpy - INFO - Chain [1] done processing
12:39:09 - cmdstanpy - INFO - Chain [1] start processing
12:39:09 - cmdstanpy - INFO - Chain [1] done processing
12:39:09 - cmdstanpy - INFO - Chain [1] start processing
12:39:09 - cmdstanpy - INFO - Chain [1] done processing
12:39:09 - cmdstanpy - INFO - Chain [1] start processing
12:39:09 - cmdstanpy - INFO - Chain [1] done processing
12:39:09 - cmdstanpy - INFO - Chain [1] start processing
12:39:09 - cmdstanpy - INFO - Chain [1] done processing
12:39:09 - cmdstanpy - INFO - Chain [1] start processing
12:39:09 - cmdstanpy - INFO - Chain [1]

include_2026=True | horizon=6m | walk-forward steps=34
RMSE: 165.4884
MAE:  125.0380
R2:   0.2461


12:41:34 - cmdstanpy - INFO - Chain [1] start processing
12:41:34 - cmdstanpy - INFO - Chain [1] done processing
12:41:34 - cmdstanpy - INFO - Chain [1] start processing
12:41:34 - cmdstanpy - INFO - Chain [1] done processing
12:41:34 - cmdstanpy - INFO - Chain [1] start processing
12:41:34 - cmdstanpy - INFO - Chain [1] done processing
12:41:34 - cmdstanpy - INFO - Chain [1] start processing
12:41:34 - cmdstanpy - INFO - Chain [1] done processing
12:41:34 - cmdstanpy - INFO - Chain [1] start processing
12:41:34 - cmdstanpy - INFO - Chain [1] done processing
12:41:34 - cmdstanpy - INFO - Chain [1] start processing
12:41:34 - cmdstanpy - INFO - Chain [1] done processing
12:41:34 - cmdstanpy - INFO - Chain [1] start processing
12:41:34 - cmdstanpy - INFO - Chain [1] done processing
12:41:34 - cmdstanpy - INFO - Chain [1] start processing
12:41:34 - cmdstanpy - INFO - Chain [1] done processing
12:41:34 - cmdstanpy - INFO - Chain [1] start processing
12:41:34 - cmdstanpy - INFO - Chain [1]

include_2026=False | horizon=6m | walk-forward steps=32
RMSE: 104.4706
MAE:  89.5593
R2:   0.3686


In [123]:
# Initialize a dictionary to store results for each horizon
all_diesel_results = {}

# Loop through forecast horizons from 1 to 6
for horizon in range(1, 7):
    print(f"Running Prophet model with FORECAST_HORIZON = {horizon}")
    FORECAST_HORIZON = horizon  # Update the global forecast horizon
    exclude_cols = ["Date", "Total_Production"]
    candidate_features = [c for c in diesel_df.columns if c not in exclude_cols]

    selected_diesel_features, final_rmse = select_prophet_features(diesel_df, candidate_features)

    # Run the model with and without 2026 data
    results_with_2026 = run_prophet_model(diesel_df, selected_diesel_features, include_2026=True, plot=False)
    results_no_2026 = run_prophet_model(diesel_df, selected_diesel_features, include_2026=False, plot=False)

    # Store the results in the dictionary
    all_diesel_results[horizon] = {
        "with_2026": results_with_2026,
        "no_2026": results_no_2026,
        "features": selected_diesel_features
    }

08:44:04 - cmdstanpy - INFO - Chain [1] start processing


Running Prophet model with FORECAST_HORIZON = 1


08:44:04 - cmdstanpy - INFO - Chain [1] done processing
08:44:04 - cmdstanpy - INFO - Chain [1] start processing
08:44:04 - cmdstanpy - INFO - Chain [1] done processing
08:44:04 - cmdstanpy - INFO - Chain [1] start processing
08:44:04 - cmdstanpy - INFO - Chain [1] done processing
08:44:04 - cmdstanpy - INFO - Chain [1] start processing
08:44:04 - cmdstanpy - INFO - Chain [1] done processing
08:44:04 - cmdstanpy - INFO - Chain [1] start processing
08:44:04 - cmdstanpy - INFO - Chain [1] done processing
08:44:04 - cmdstanpy - INFO - Chain [1] start processing
08:44:04 - cmdstanpy - INFO - Chain [1] done processing
08:44:04 - cmdstanpy - INFO - Chain [1] start processing
08:44:04 - cmdstanpy - INFO - Chain [1] done processing
08:44:04 - cmdstanpy - INFO - Chain [1] start processing
08:44:04 - cmdstanpy - INFO - Chain [1] done processing
08:44:04 - cmdstanpy - INFO - Chain [1] start processing
08:44:04 - cmdstanpy - INFO - Chain [1] done processing
08:44:04 - cmdstanpy - INFO - Chain [1] 

include_2026=True | horizon=1m | walk-forward steps=36
RMSE: 116.8446
MAE:  70.3609
R2:   0.8542


08:47:29 - cmdstanpy - INFO - Chain [1] done processing
08:47:29 - cmdstanpy - INFO - Chain [1] start processing
08:47:29 - cmdstanpy - INFO - Chain [1] done processing
08:47:29 - cmdstanpy - INFO - Chain [1] start processing
08:47:29 - cmdstanpy - INFO - Chain [1] done processing
08:47:29 - cmdstanpy - INFO - Chain [1] start processing
08:47:29 - cmdstanpy - INFO - Chain [1] done processing
08:47:29 - cmdstanpy - INFO - Chain [1] start processing
08:47:29 - cmdstanpy - INFO - Chain [1] done processing
08:47:29 - cmdstanpy - INFO - Chain [1] start processing
08:47:29 - cmdstanpy - INFO - Chain [1] done processing
08:47:29 - cmdstanpy - INFO - Chain [1] start processing
08:47:29 - cmdstanpy - INFO - Chain [1] done processing
08:47:29 - cmdstanpy - INFO - Chain [1] start processing
08:47:29 - cmdstanpy - INFO - Chain [1] done processing
08:47:29 - cmdstanpy - INFO - Chain [1] start processing
08:47:29 - cmdstanpy - INFO - Chain [1] done processing
08:47:30 - cmdstanpy - INFO - Chain [1] 

...

include_2026=False | horizon=5m | walk-forward steps=33
RMSE: 195.5227
MAE:  158.5327
R2:   -0.4810
Running Prophet model with FORECAST_HORIZON = 6


08:55:34 - cmdstanpy - INFO - Chain [1] start processing
08:55:34 - cmdstanpy - INFO - Chain [1] done processing
08:55:34 - cmdstanpy - INFO - Chain [1] start processing
08:55:34 - cmdstanpy - INFO - Chain [1] done processing
08:55:34 - cmdstanpy - INFO - Chain [1] start processing
08:55:34 - cmdstanpy - INFO - Chain [1] done processing
08:55:34 - cmdstanpy - INFO - Chain [1] start processing
08:55:34 - cmdstanpy - INFO - Chain [1] done processing
08:55:34 - cmdstanpy - INFO - Chain [1] start processing
08:55:34 - cmdstanpy - INFO - Chain [1] done processing
08:55:34 - cmdstanpy - INFO - Chain [1] start processing
08:55:34 - cmdstanpy - INFO - Chain [1] done processing
08:55:34 - cmdstanpy - INFO - Chain [1] start processing
08:55:34 - cmdstanpy - INFO - Chain [1] done processing
08:55:34 - cmdstanpy - INFO - Chain [1] start processing
08:55:34 - cmdstanpy - INFO - Chain [1] done processing
08:55:34 - cmdstanpy - INFO - Chain [1] start processing
08:55:34 - cmdstanpy - INFO - Chain [1]

include_2026=True | horizon=6m | walk-forward steps=34
RMSE: 302.2192
MAE:  207.6060
R2:   0.0661


08:56:49 - cmdstanpy - INFO - Chain [1] start processing
08:56:49 - cmdstanpy - INFO - Chain [1] done processing
08:56:49 - cmdstanpy - INFO - Chain [1] start processing
08:56:49 - cmdstanpy - INFO - Chain [1] done processing
08:56:49 - cmdstanpy - INFO - Chain [1] start processing
08:56:49 - cmdstanpy - INFO - Chain [1] done processing
08:56:49 - cmdstanpy - INFO - Chain [1] start processing
08:56:49 - cmdstanpy - INFO - Chain [1] done processing
08:56:49 - cmdstanpy - INFO - Chain [1] start processing
08:56:49 - cmdstanpy - INFO - Chain [1] done processing
08:56:49 - cmdstanpy - INFO - Chain [1] start processing
08:56:49 - cmdstanpy - INFO - Chain [1] done processing
08:56:49 - cmdstanpy - INFO - Chain [1] start processing
08:56:49 - cmdstanpy - INFO - Chain [1] done processing
08:56:49 - cmdstanpy - INFO - Chain [1] start processing
08:56:49 - cmdstanpy - INFO - Chain [1] done processing
08:56:49 - cmdstanpy - INFO - Chain [1] start processing
08:56:49 - cmdstanpy - INFO - Chain [1]

include_2026=False | horizon=6m | walk-forward steps=32
RMSE: 196.0358
MAE:  156.0651
R2:   -0.4462


Current findings:
Summary of Results:
====================================================================
| Horizon | 2026 included | RMSE     | MAE      | R^2      |
====================================================================
| 1       | True           | 116.8446 | 70.3609  | 0.8542   |
| 1       | False          | 62.2736  | 48.8523  | 0.8440   |
| 2       | True           | 204.2396 | 127.0232 | 0.5546   |
| 2       | False          | 123.6725 | 94.5154  | 0.3982   |
| 3       | True           | 300.4642 | 192.0415 | 0.0361   |
| 3       | False          | 191.9307 | 154.2861 | -0.4493  |
| 4       | True           | 337.1029 | 224.1892 | -0.1816  |
| 4       | False          | 218.3076 | 175.8406 | -0.8750  |
| 5       | True           | 340.0624 | 240.9044 | -0.2024  |
| 5       | False          | 225.5254 | 187.7813 | -1.0011  |
| 6       | True           | 336.6316 | 252.2795 | -0.1783  |
| 6       | False          | 222.2204 | 189.2934 | -0.9428  |
====================================================================

With Features:
['BFP',
 'Brent_LastWeekMean',
 'Brent_LastWeekMean_Lag2',
 'Total_Production_Lag1',
 'USDZAR_LastWeekMean',
 'USDZAR_Mean_Lag1',
 'BFP_Delta_Lag5',
 'BFP_Delta_Lag1',
 'BFP_Lag1',
 'Brent_MonthMean_Lag1']

In [100]:
selected_diesel_features

['BFP',
 'Brent_LastWeekMean',
 'Brent_LastWeekMean_Lag2',
 'Total_Production_Lag1',
 'USDZAR_LastWeekMean',
 'USDZAR_Mean_Lag1',
 'BFP_Delta_Lag5',
 'BFP_Delta_Lag1',
 'BFP_Lag1',
 'Brent_MonthMean_Lag1']

In [115]:
# Print all results at the end
print("\nSummary of Results:")
print("====================================================================")
print("| Horizon | 2026 included | RMSE     | MAE      | R^2      |")
print("====================================================================")
for horizon, results in all_results.items():
    for include_2026, result_key in zip([True, False], ["with_2026", "no_2026"]):
        result = results[result_key]
        print(f"| {horizon:<7} | {str(include_2026):<14} | {result['rmse']:<8.4f} | {result['mae']:<8.4f} | {result['r2']:<8.4f} |")
print("====================================================================")


Summary of Results:
| Horizon | 2026 included | RMSE     | MAE      | R^2      |
| 1       | True           | 116.8446 | 70.3609  | 0.8542   |
| 1       | False          | 62.2736  | 48.8523  | 0.8440   |
| 2       | True           | 193.3446 | 126.5257 | 0.6113   |
| 2       | False          | 119.7145 | 99.5984  | 0.4448   |
| 3       | True           | 317.5438 | 192.9060 | -0.0310  |
| 3       | False          | 185.9549 | 148.2243 | -0.3396  |
| 4       | True           | 319.5041 | 188.0659 | -0.0438  |
| 4       | False          | 208.4546 | 155.7920 | -0.6834  |
| 5       | True           | 295.0028 | 204.1233 | 0.1102   |
| 5       | False          | 195.5227 | 158.5327 | -0.4810  |
| 6       | True           | 302.2192 | 207.6060 | 0.0661   |
| 6       | False          | 196.0358 | 156.0651 | -0.4462  |
